In [ ]:
import sys

!{sys.executable} -m pip install --no-cache-dir --force-reinstall "typing_extensions==4.16.0"

In [ ]:
!pip install -q mlflow opencv-python-headless seaborn scikit-learn

In [ ]:
import subprocess
import socket
import sys
import time
from functools import lru_cache
from pathlib import Path
from types import SimpleNamespace

import cv2
import keras
import matplotlib.pyplot as plt
import mlflow
import mlflow.keras
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
from keras.utils import to_categorical
from mlflow import MlflowClient
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
print("TensorFlow :", tf.__version__, "| Keras :", keras.__version__)
print("GPU détecté :", tf.config.list_physical_devices("GPU"))

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "abderrahmanoirgari/neuro-ai2",
    output_dir="/workspace/kaggle_download"
)

print("Dataset téléchargé dans :", dataset_path)

In [ ]:
from pathlib import Path
import shutil

WORK = Path("/workspace")
SOURCE_DIR = WORK / "kaggle_download" / "Data"
RAW_DIR = WORK / "data" / "raw"

# Vérifier que le dossier source existe
assert SOURCE_DIR.is_dir(), f"Dossier source introuvable : {SOURCE_DIR}"

# Créer le dossier de destination
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Copier les quatre dossiers de classes
for class_dir in SOURCE_DIR.iterdir():
    if class_dir.is_dir():
        destination = RAW_DIR / class_dir.name
        shutil.copytree(class_dir, destination, dirs_exist_ok=True)

print("Classes copiées :", sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir()))

In [ ]:
WORK = Path("/workspace")                    # dossier de travail sur la VM (à adapter si besoin)
RAW_DIR = WORK / "data" / "raw"
print("Classes trouvées :", sorted(p.name for p in RAW_DIR.iterdir()))

config = SimpleNamespace(
    # Chemins
    RAW_DIR=RAW_DIR,
    CACHE_FILE_RGB=WORK / "processed" / "dataset_rgb.npz",
    MODELS_DIR=WORK / "models",
    REPORTS_DIR=WORK / "reports",
    # Données
    CLASSES=["glioma", "meningioma", "notumor", "pituitary"],
    VALID_EXT={".jpeg", ".jpg", ".bmp", ".png"},
    IMG_SIZE=224,
    CHANNELS=3,
    # Entraînement
    SEED=42,
    BATCH_SIZE=16,
    LEARNING_RATE=1e-3,
    SCRATCH_EPOCHS=30,
    TL_EPOCHS=15, TL_LR=1e-3, TL_DROPOUT=0.3,
    FT_EPOCHS=20, FT_LR=1e-4, FT_UNFREEZE_FROM=100,
    # MLflow
    MLFLOW_URI="http://127.0.0.1:5000",
    EXPERIMENT_NAME="neurovision-part2",
    REGISTERED_MODEL="neurovision-classifier",
    CHAMPION_ALIAS="champion",
)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight


def load_images_rgb(raw_dir=config.RAW_DIR, img_size=config.IMG_SIZE):
    images, labels = [], []
    skipped = 0

    for class_dir in sorted(raw_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name


        for file in sorted(class_dir.iterdir()):
            if file.suffix.lower() not in config.VALID_EXT:
                skipped += 1
                continue
            try:
                img = cv2.imread(str(file), cv2.IMREAD_COLOR)
                if img is None:
                    raise ValueError("image illisible")
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
            except Exception as e:
                print(f"[ignorée] {file.name} : {e}")
                skipped += 1
                continue

            images.append(img)
            labels.append(label)

    print(f"{len(images)} images chargées, {skipped} ignorées.")

    images, labels = shuffle(images, labels, random_state=config.SEED)
    return np.array(images, dtype="uint8"), np.array(labels)


def load_dataset_rgb(use_cache=True):
    if use_cache and config.CACHE_FILE_RGB.exists():
        data = np.load(config.CACHE_FILE_RGB)
        return data["X"], data["y"]

    X, y = load_images_rgb()
    config.CACHE_FILE_RGB.parent.mkdir(parents=True, exist_ok=True)
    np.savez(config.CACHE_FILE_RGB, X=X, y=y)
    return X, y


def encode_labels(y):
    return LabelEncoder().fit(config.CLASSES).transform(y)


def get_splits():
    X, y = load_dataset_rgb()
    y_int = encode_labels(y)

    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y_int, test_size=0.30, stratify=y_int, random_state=config.SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=config.SEED)

    n = len(config.CLASSES)
    return (X_train, to_categorical(y_train, n),
            X_val, to_categorical(y_val, n),
            X_test, to_categorical(y_test, n))


def get_class_weights(y_train_onehot):
    y_int = np.argmax(y_train_onehot, axis=1)
    classes = np.unique(y_int)
    weights = compute_class_weight("balanced", classes=classes, y=y_int)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def make_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.cache()
    if training:
        ds = ds.shuffle(len(X), seed=config.SEED)
    ds = ds.map(lambda img, lbl: (tf.cast(img, tf.float32), lbl),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(config.BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
import keras
import numpy as np
from keras import layers



def _augmentation():
    return keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.05, 0.05),
    ], name="augmentation")


def build_scratch_cnn():

    inputs = keras.Input(shape=(config.IMG_SIZE, config.IMG_SIZE, config.CHANNELS))
    x = _augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(config.CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs, name="cnn_from_scratch")

def build_transfer_model():
    base = keras.applications.MobileNetV2(
        input_shape=(config.IMG_SIZE, config.IMG_SIZE, config.CHANNELS),
        include_top=False,
        weights="imagenet",
        name="mobilenetv2_base",
    )
    base.trainable = False

    inputs = keras.Input(shape=(config.IMG_SIZE, config.IMG_SIZE, config.CHANNELS))
    x = _augmentation()(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(config.TL_DROPOUT)(x)
    outputs = layers.Dense(len(config.CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2_transfer")

def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def unfreeze_for_finetuning(model, unfreeze_from=config.FT_UNFREEZE_FROM):
    base = model.get_layer("mobilenetv2_base")
    base.trainable = True
    for layer in base.layers[:unfreeze_from]:
        layer.trainable = False
    for layer in base.layers[unfreeze_from:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    return model

def count_parameters(model):
    trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    total = int(model.count_params())
    return {"total_params": total,
            "trainable_params": trainable,
            "non_trainable_params": total - trainable}

In [ ]:
import mlflow
import mlflow.keras
from mlflow import MlflowClient



def setup_mlflow():
    mlflow.set_tracking_uri(config.MLFLOW_URI)
    mlflow.set_experiment(config.EXPERIMENT_NAME)


def _experiment_id():
    return mlflow.get_experiment_by_name(config.EXPERIMENT_NAME).experiment_id


def compare_runs():
    df = mlflow.search_runs(
        experiment_ids=[_experiment_id()],
        filter_string="attributes.status = 'FINISHED'",
        order_by=["metrics.best_val_f1_macro DESC"],
    )
    cols = [
        "tags.mlflow.runName",
        "params.learning_rate",
        "params.trainable_params",
        "metrics.best_val_accuracy",
        "metrics.best_val_f1_macro",
        "metrics.best_val_loss",
        "metrics.duration_min",
    ]
    return df[[c for c in cols if c in df.columns]]


def get_run_model_uri(run_name):
    runs = mlflow.search_runs(
        experiment_ids=[_experiment_id()],
        filter_string=f"tags.mlflow.runName = '{run_name}' and attributes.status = 'FINISHED'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    if runs.empty:
        raise ValueError(f"Aucun run terminé nommé '{run_name}'.")
    return runs.iloc[0]["tags.model_uri"]


def promote_champion(metric="best_val_f1_macro"):
    runs = mlflow.search_runs(
        experiment_ids=[_experiment_id()],
        filter_string="attributes.status = 'FINISHED'",
        order_by=[f"metrics.{metric} DESC"],
        max_results=1,
    )
    if runs.empty:
        raise ValueError("Aucun run terminé.")
    best = runs.iloc[0]

    version = mlflow.register_model(best["tags.model_uri"], config.REGISTERED_MODEL)
    MlflowClient().set_registered_model_alias(
        config.REGISTERED_MODEL, config.CHAMPION_ALIAS, version.version)
    print(f"Champion : run '{best['tags.mlflow.runName']}' -> "
          f"{config.REGISTERED_MODEL} v{version.version} (alias '{config.CHAMPION_ALIAS}')")
    return best

In [ ]:
import time

import keras
import matplotlib.pyplot as plt
import mlflow
import mlflow.keras
import numpy as np
from sklearn.metrics import f1_score



class MlflowEpochLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        mlflow.log_metrics({k: float(v) for k, v in (logs or {}).items()}, step=epoch)


def plot_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history[f"val_{metric}"], label="validation")
        ax.set_title(metric)
        ax.set_xlabel("époque")
        ax.legend()
    return fig


def train_and_log(model, run_name, train_ds, val_ds, y_val, class_weights,
                  epochs, learning_rate, patience=5, extra_params=None):

    config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    ckpt = config.MODELS_DIR / f"{run_name}.keras"
    backup = config.MODELS_DIR / f"backup_{run_name}"

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            str(ckpt),
            monitor="val_loss",
            save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        ),
        keras.callbacks.BackupAndRestore(
            backup_dir=str(backup)
        ),
        MlflowEpochLogger(),
    ]

    summary_lines = []
    model.summary(print_fn=summary_lines.append)

    with mlflow.start_run(run_name=run_name):
        # hyperparamètres
        mlflow.log_params({
            "model": model.name, "learning_rate": learning_rate, "epochs_max": epochs,
            "batch_size": config.BATCH_SIZE, "img_size": config.IMG_SIZE,
            "optimizer": "Adam", "loss": "categorical_crossentropy",
            **count_parameters(model), **(extra_params or {}),
        })
        mlflow.log_text("\n".join(summary_lines), "model_summary.txt")

        # entraînement chronométré
        start = time.time()
        history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                            class_weight=class_weights, callbacks=callbacks)
        duration_min = (time.time() - start) / 60
        print(f"Durée d'entraînement : {duration_min:.1f} min")

        # métriques du MEILLEUR modèle sur la validation
        best_model = keras.models.load_model(ckpt)
        val_loss, val_acc = best_model.evaluate(val_ds, verbose=0)
        y_true = np.argmax(y_val, axis=1)
        y_pred = np.argmax(best_model.predict(val_ds, verbose=0), axis=1)
        val_f1 = f1_score(y_true, y_pred, average="macro")
        print(f"Validation -> loss {val_loss:.4f} | acc {val_acc:.4f} | F1 macro {val_f1:.4f}")

        mlflow.log_metrics({"duration_min": duration_min, "best_val_loss": val_loss,
                            "best_val_accuracy": val_acc, "best_val_f1_macro": val_f1})
        mlflow.log_param("epochs_run", len(history.history["loss"]))

        # artefacts : courbes + modèle
        mlflow.log_figure(plot_curves(history), "learning_curves.png")
        info = mlflow.keras.log_model(best_model, name="model")
        mlflow.set_tag("model_uri", info.model_uri)       # retrouvé par tracking.py

    return best_model, history

In [ ]:
def mlflow_is_up(port=5000):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


def ensure_mlflow_server():
    if mlflow_is_up():
        print("Serveur MLflow déjà lancé.")
        return
    subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server",
         "--host", "127.0.0.1", "--port", "5000",
         "--backend-store-uri", f"sqlite:///{WORK / 'mlflow.db'}",
         "--artifacts-destination", str(WORK / "mlartifacts")],
        stdout=open(WORK / "mlflow_server.log", "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if mlflow_is_up():
            print("Serveur MLflow lancé.")
            return
        time.sleep(1)
    raise RuntimeError("Le serveur MLflow ne démarre pas : regarde /workspace/mlflow_server.log")


ensure_mlflow_server()
setup_mlflow()

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score


def evaluate_on_test(model, X_test, y_test, test_ds, run_id=None):
    y_true = np.argmax(y_test, axis=1)
    y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

    print(classification_report(y_true, y_pred, target_names=config.CLASSES, digits=3))


    cm = confusion_matrix(y_true, y_pred)
    fig_cm, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=config.CLASSES, yticklabels=config.CLASSES)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title("Matrice de confusion (test)")

    # Faux négatifs graves : une vraie tumeur prédite "notumor"
    tumor_ids = [config.CLASSES.index(c) for c in ("glioma", "meningioma", "pituitary")]
    notumor_id = config.CLASSES.index("notumor")
    is_tumor = np.isin(y_true, tumor_ids)
    missed = int(np.sum(is_tumor & (y_pred == notumor_id)))
    missed_rate = missed / int(is_tumor.sum())
    print(f"Tumeurs prédites 'notumor' (faux négatifs graves) : "
          f"{missed}/{int(is_tumor.sum())} ({missed_rate:.2%})")

    # Quelques erreurs
    wrong = np.where(y_true != y_pred)[0][:8]
    fig_err = None
    if len(wrong):
        fig_err, axes = plt.subplots(2, 4, figsize=(12, 6))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i in zip(axes.ravel(), wrong):
            ax.imshow(X_test[i].astype("uint8"))
            ax.set_title(f"réel: {config.CLASSES[y_true[i]]}\nprédit: {config.CLASSES[y_pred[i]]}",
                         fontsize=9)


    recalls = cm.diagonal() / cm.sum(axis=1)

    metrics = {
        "test_accuracy": float(np.mean(y_true == y_pred)),
        "test_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "test_tumor_missed_rate": float(missed_rate),
        **{
            f"test_recall_{c}": float(r)
            for c, r in zip(config.CLASSES, recalls)
        },
    }

    print(metrics)

In [ ]:
from functools import lru_cache
from pathlib import Path

import cv2
import mlflow
import mlflow.keras
import numpy as np


@lru_cache(maxsize=1)
def get_champion():
    mlflow.set_tracking_uri(config.MLFLOW_URI)
    uri = f"models:/{config.REGISTERED_MODEL}@{config.CHAMPION_ALIAS}"
    try:
        return mlflow.keras.load_model(uri)
    except Exception as e:
        raise RuntimeError(
            f"Impossible de charger '{uri}'. Le serveur MLflow est-il lancé sur {config.MLFLOW_URI} ? "
            f"({e})")


def predict_array(img_rgb):
    img = cv2.resize(img_rgb, (config.IMG_SIZE, config.IMG_SIZE)).astype("float32")
    img = img[np.newaxis, ...]

    probs = get_champion().predict(img, verbose=0)[0]
    best = int(np.argmax(probs))
    return config.CLASSES[best], float(probs[best]), dict(zip(config.CLASSES, map(float, probs)))


def predict_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    if path.suffix.lower() not in config.VALID_EXT:
        raise ValueError(f"Extension non supportée : {path.suffix}")

    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Image illisible ou corrompue.")
    return predict_array(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))




In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = get_splits()
class_weights = get_class_weights(y_train)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)

In [ ]:
from pathlib import Path
import subprocess
import sys
import time
import socket

WORK = Path("/workspace")
(WORK / "models").mkdir(parents=True, exist_ok=True)
(WORK / "reports").mkdir(parents=True, exist_ok=True)
(WORK / "processed").mkdir(parents=True, exist_ok=True)

def mlflow_is_up(port=5000):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

if mlflow_is_up():
    print("MLflow est déjà actif.")
else:
    log_file = open(WORK / "mlflow_server.log", "a")
    subprocess.Popen(
        [
            sys.executable, "-m", "mlflow", "server",
            "--host", "127.0.0.1",
            "--port", "5000",
            "--backend-store-uri",
            f"sqlite:///{WORK / 'mlflow.db'}",
            "--artifacts-destination",
            str(WORK / "mlartifacts"),
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    for _ in range(30):
        if mlflow_is_up():
            print("MLflow a démarré avec succès.")
            break
        time.sleep(1)
    else:
        raise RuntimeError(
            "MLflow ne démarre pas. Consulte /workspace/mlflow_server.log"
        )

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.get_experiment_by_name("neurovision-part2")

if experiment is None:
    print("L'expérience neurovision-part2 n'existe pas encore.")
else:
    print("Expérience :", experiment.name)
    print("ID :", experiment.experiment_id)

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )
    print("Nombre de runs :", len(runs))
    if not runs.empty:
        print(runs[["tags.mlflow.runName", "status"]])

In [ ]:
model = mlflow.keras.load_model(get_run_model_uri("transfer_learning"))
compare_runs()           # performances AVANT fine-tuning

In [ ]:
unfreeze_for_finetuning(model)                       # base.layers[100:] dégelées (sauf BatchNorm)
compile_model(model, config.FT_LR)                   # lr 1e-4 : recompilation obligatoire

base = model.get_layer("mobilenetv2_base")
print(f"Couches de la base entraînables : {sum(l.trainable for l in base.layers)} / {len(base.layers)}")
print(count_parameters(model))
model.summary()

In [ ]:
ft_best, ft_hist = train_and_log(
    model, "fine_tuning", train_ds, val_ds, y_val, class_weights,
    epochs=config.FT_EPOCHS, learning_rate=config.FT_LR, patience=5,
    extra_params={"phase": "fine_tuning", "unfreeze_from": config.FT_UNFREEZE_FROM},
)

In [ ]:
compare_runs()

In [ ]:
best = promote_champion()          # meilleur F1 de VALIDATION -> Model Registry + alias 'champion'

In [ ]:
champion = get_champion()                           # chargé depuis MLflow via l'alias
test_ds = make_dataset(X_test, y_test)
evaluate_on_test(champion, X_test, y_test, test_ds, run_id=best["run_id"])

In [ ]:
test_path = sorted((config.RAW_DIR / "glioma").iterdir())[0]
try:
    label, conf, probs = predict_image(test_path)
    print(f"{label} ({conf:.1%})")
except Exception as e:
    print("Erreur :", e)

In [ ]:
import zipfile

with zipfile.ZipFile(WORK / "mlflow_export.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(WORK / "mlflow.db", "mlflow.db")
    for p in (WORK / "mlartifacts").rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
print("OK :", WORK / "mlflow_export.zip")